# B1.0 · What a harness is, and the loop it runs

**Function B — Application Security with an AI SDLC → What Runs the Pipeline**  ·  *Both directions*

Builds on **[A3.10 · The agent's escalation path](https://spbreed.github.io/cyber-commons/lessons/A3.10.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A pipeline whose verifier is the model agreeing with itself does not fail loudly — it succeeds incorrectly, files a clean trace, and the bug is found by whoever merged the patch. Seven components, four moves, and the one nobody can name is almost always the verifier.

> **At CyberTravels.** CyberTravels already runs four harnesses and calls them agents. The Workflow Agent is a loop over booking tools whose verifier nobody ever specified — which is the answer to why it refunded twice.

## 2 · The framework

```
   the eight components of any harness

   +-----------+   +--------+   +---------+   +-----------+
   |   model   |   |  loop  |   |  tools  |   |  context  |
   +-----------+   +--------+   +---------+   +-----------+
   +-----------+   +--------+   +---------+   +-----------+
   | verifier  |   | budget |   | memory  |   |orchestrator|
   +-----------+   +--------+   +---------+   +-----------+
                                              (+ telemetry)

   the one people cannot name is almost always the verifier
   -- and when something goes wrong, the class of failure
      is just "which of these eight did it"
```

**The model is not the system.**

A model is a text generator. Give it tokens, get tokens back. It has no memory
between calls, no ability to act, and no notion of whether it succeeded. Left
alone it cannot read a file, run a scanner or open a pull request.

> **A harness is everything wrapped around a model that turns generating text
> into getting work done.** It decides what the model sees, what it may do,
> whether what it did worked, when to stop, and what is written down
> afterwards.

Every stage of the pipeline in chapter 5 is one of these. So is each of
CyberTravels' four agents. Being able to name the parts is what lets you say
which part failed.

| Component | What it does |
|---|---|
| **The loop** | Decides what happens next, and when to stop |
| **Tools** | The only way the model touches the world |
| **Context** | What the model sees at each step, assembled from a world much larger than the window |
| **The verifier** | The independent check on whether a step actually succeeded |
| **State** | What survives between steps and between runs |
| **Budget** | Token, time, cost and action ceilings that bound autonomy |
| **Telemetry** | The record that makes a run auditable and replayable |

### The loop is four moves

**Plan** — the model proposes what to do next. **Act** — the harness executes
that proposal against a tool. **Verify** — something decides whether the result
is acceptable. **Stop** — either verification succeeded or a budget ran out.

That is the whole architecture, and everything that makes a pipeline
trustworthy lives in moves 3 and 4. Frameworks make moves 1 and 2 easy and
leave 3 and 4 as your problem, usually defaulting to "the model says it's done"
and "loop forever".

### The verifier decides what the pipeline may conclude

State it plainly, because chapter 5 depends on it. **A harness with a weak
verifier does not fail loudly. It succeeds incorrectly**, produces a clean
trace, and the failure is found downstream — by the reviewer who merged the
patch, or the traveller who was refunded twice.

Verifiers form a hierarchy, ordered by what it takes to fool them:

| Verifier | Fooled by | Available when |
|---|---|---|
| **Behavioural test** | changing real behaviour | you can execute the thing |
| **Exact-match oracle** | nothing, but needs the answer up front | rarely |
| **Shape check** | any well-formed output | always |
| **LLM judge** | confident prose | always |

The trap is that the two available everywhere are the two weakest, and they
fail in the worst direction: they do not error, they **approve**. A pipeline
whose verifier is the model agreeing with itself produces confident nonsense at
the rate the model produces anything.

## 3 · Build the smallest thing that is still a harness

Seven components, wired in about forty lines. The model is a deterministic stub so the *harness* is what you are looking at.

In [ ]:
from dataclasses import dataclass

def stand_in_model(prompt):
    """NOT a language model. A deterministic stub, so the harness is visible.

    It reads the transcript so far to decide what is left to do - which is all
    any agent loop does, minus the part that is hard."""
    if "write_patch" not in prompt:
        return {"tool": "write_patch", "args": {"file": "refunds.py"}}
    if "run_tests" not in prompt:
        return {"tool": "run_tests", "args": {}}
    return {"tool": "done", "args": {"claim": "fixed it"}}

WORLD = {"tests_pass": False, "patched": False}

def run_tests(**_):
    # the patch this stub writes does not actually fix the bug
    return {"passed": WORLD["tests_pass"],
            "failing": [] if WORLD["tests_pass"] else ["test_refund_window"]}
def write_patch(file, **_):
    WORLD["patched"] = True
    return {"wrote": file}
def done(claim, **_):
    return {"claim": claim}

TOOLS = {"run_tests": run_tests, "write_patch": write_patch, "done": done}

@dataclass
class Budget:
    steps: int = 6
    used: int = 0
    def spend(self):
        self.used += 1
        return self.used <= self.steps

def harness(task, verifier=None, budget=None, telemetry=None):
    """loop + tools + context + verifier + state + budget + telemetry."""
    budget = budget or Budget()
    telemetry = telemetry if telemetry is not None else []
    context = [f"TASK: {task}"]                       # context
    state = {"steps": 0}                              # state
    while budget.spend():                             # budget / stop condition
        step = stand_in_model("\n".join(context))     # the model
        tool, args = step["tool"], step["args"]
        result = TOOLS[tool](**args)                  # tools
        state["steps"] += 1
        telemetry.append({"step": state["steps"], "tool": tool, "result": result})
        context.append(f"{tool} -> {result}")
        if tool == "done":
            ok = verifier() if verifier else True     # the verifier
            return {"claimed": True, "verified": ok, "steps": state["steps"],
                    "telemetry": telemetry}
    return {"claimed": False, "verified": False, "steps": state["steps"],
            "telemetry": telemetry}

print("components wired:", ["loop", "tools", "context", "verifier",
                            "state", "budget", "telemetry"])

## 4 · Run it once with no verifier

In [ ]:
WORLD.update(tests_pass=False, patched=False)
r = harness("fix the failing test_refund_window", verifier=None)
print(f"agent claimed success : {r['claimed']}")
print(f"independently checked : {r['verified']}")
for t in r["telemetry"]:
    print(f"   {t['step']}. {t['tool']:12s}{t['result']}")
print()
print("It reported success. The tests still fail. Nothing in that transcript is")
print("a lie - the agent did write a patch, and then it said it was done.")
assert r["claimed"] and not WORLD["tests_pass"]

## 5 · Add the one component that was missing

Same model, same tools, same proposals, same order. The only difference is what the loop is allowed to believe.

In [ ]:
def real_verifier():
    """Reads ground truth, not the agent's claim."""
    return run_tests()["passed"]

WORLD.update(tests_pass=False, patched=False)
r2 = harness("fix the failing test_refund_window", verifier=real_verifier)
print(f"claimed  : {r2['claimed']}")
print(f"verified : {r2['verified']}   <- the pipeline now knows")

WORLD.update(tests_pass=True)                 # a patch that genuinely works
r3 = harness("fix the failing test_refund_window", verifier=real_verifier)
print(f"\nwith a working patch -> claimed {r3['claimed']}, "
      f"verified {r3['verified']}")
print()
print("One component. Without it the pipeline files a ticket saying the bug is")
print("fixed; with it the same run is correctly reported as not fixed.")
assert r2["claimed"] and not r2["verified"] and r3["verified"]

## 6 · The budget is a security control, not a cost control

A stop condition is what turns "the agent misbehaved" into "the agent misbehaved six times". It is the only control in the loop that holds when every other one has been talked around.

In [ ]:
def looping_model(prompt):
    """A model that never emits `done` - a stuck loop, or a driven one."""
    return {"tool": "run_tests", "args": {}}

# Swap the model the loop calls. `harness` resolves `stand_in_model` at call
# time from module globals, so rebinding the name is enough - no `global`
# statement, which is a syntax error at module level anyway.
_real = stand_in_model
stand_in_model = looping_model
WORLD.update(tests_pass=False)
stuck = harness("fix it", verifier=real_verifier, budget=Budget(steps=4))
stand_in_model = _real

print(f"steps taken   : {stuck['steps']}  (ceiling was 4)")
print(f"claimed       : {stuck['claimed']}")
print()
print("The budget did not make the model behave. It made the misbehaviour")
print("finite, which is the only property available once the model is the")
print("component you cannot trust.")
assert stuck["steps"] == 4 and not stuck["claimed"]

## 7 · Four verifiers, and the two you will actually have

The loop above used the strongest kind. Most pipeline stages cannot — you cannot execute a threat model. So it is worth seeing all four against one malformed finding.

In [ ]:
FINDING = {"cwe": "CWE-89", "file": "src/data/reports.py", "line": 2,
           "severity": "high",
           "rationale": "The query is constructed safely using parameters."}
TRUTH   = {"cwe": "CWE-89", "file": "src/data/reports.py", "line": 2}

def shape_check(f):
    """Available always. Fooled by anything well-formed."""
    return all(k in f for k in ("cwe", "file", "line", "severity"))

def llm_judge(f):
    """Available always. Fooled by confident prose."""
    confident = len(f.get("rationale", "")) > 20 and "." in f["rationale"]
    return confident

def exact_oracle(f):
    """Needs the answer up front, so rarely available."""
    return (f["cwe"], f["file"], f["line"]) == (TRUTH["cwe"], TRUTH["file"],
                                                TRUTH["line"])

def behavioural(f):
    """Executes the claim: is the line actually a concatenated query?"""
    src = 'return DB.execute("SELECT * FROM bookings WHERE ref=" + ref)'
    return "+" in src and "execute" in src

for name, fn in (("shape check", shape_check), ("LLM judge", llm_judge),
                 ("exact-match oracle", exact_oracle),
                 ("behavioural test", behavioural)):
    print(f"   {name:20s}{'ACCEPTS' if fn(FINDING) else 'refuses'}")

print()
print("The finding IS a real SQL injection, and its rationale says the exact")
print("opposite - it claims the query is parameterised. The shape check accepts")
print("it because every key is present. The judge accepts it because it reads")
print("like an explanation. Neither of them read the code.")
assert shape_check(FINDING) and llm_judge(FINDING) and behavioural(FINDING)

## 8 · The harness is itself an actor

It holds credentials, calls tools and reads untrusted input. Every risk in Function A applies to it, and being a security tool grants no exemption — which is why chapter 5 closes on injection in the pipeline and on the coding agents that feed it.

In [ ]:
HARNESS_ACTOR = {
 "identity": "spiffe://cybertravels.com/ns/ci/sa/review-pipeline",
 "scopes":   {"repo:read", "repo:comment"},        # NOT repo:write
 "reads":    ["the diff", "the PR description", "commit messages",
              "code comments", "test fixtures"],
 "telemetry": "every tool call, with the span that motivated it",
}
print("the pipeline, described the way Function A describes an agent:")
for k, v in HARNESS_ACTOR.items():
    print(f"   {k:11s}{v}")
print()
wanted = {"repo:write"}
print(f"could it merge its own fix? {bool(wanted & HARNESS_ACTOR['scopes'])}")
print("Everything in the `reads` list is written by whoever opened the pull")
print("request. That is A1.9, and the pipeline reads it by definition.")
assert not (wanted & HARNESS_ACTOR["scopes"])

## What you just proved

The minimal harness reports success while the tests still fail. Adding one component — a verifier that reads ground truth rather than the agent's claim — reports the same run as unverified, and verifies the run where the patch genuinely works. A budget of four stops a model that never emits `done`. Against a finding whose rationale contradicts the finding itself, the shape check and the LLM judge both accept it. The pipeline's own identity holds `repo:comment` and not `repo:write`.

## Your turn

Name your pipeline's verifier out loud. If the sentence contains "the model checks" or "it looks right", you have a judge — and a judge approves confident prose, including prose that contradicts the finding it is attached to.

---

**Next → [B1.1 · Who chooses the next tool call — you, the model, or a server](https://spbreed.github.io/cyber-commons/lessons/B1.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.0.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.0.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*